<a href="https://colab.research.google.com/github/Luaalmed/Aula-02-Intelig-ncia-Artificial/blob/main/AG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Algoritmo Genético - *Knapsack Problem*

In [1]:
!pip install pyeasyga pyswarms


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.4 MB/s eta 0:00:00
  Created wheel for pyeasyga: filename=pyeasyga-0.3.1-py2.py3-none-any.whl size=6784 sha256=961880677f21e8efe1899587f4a05df60af55ba7ae5dd82a14bf09e09672fa5a
  Stored in directory: /root/.cache/pip/wheels/5b/cb/8f/1f54efc0a60a5c5ea25349372a2c108415b214375b0aa276f7
Successfully built pyeasyga


In [2]:
from pyeasyga import pyeasyga
import random

data = [{'name': 'verde', 'value': 4, 'weight': 12},
        {'name': 'cinza', 'value': 2, 'weight': 1},
        {'name': 'amarelo', 'value': 10, 'weight': 4},
        {'name': 'laranja', 'value': 1, 'weight': 1},
        {'name': 'azul', 'value': 2, 'weight': 2}]

tamanho_populacao = 20
geracoes = 50

ga = pyeasyga.GeneticAlgorithm(data,
                               population_size = tamanho_populacao,
                               generations = geracoes,
                               crossover_probability = 0.9,
                               mutation_probability = 0.3,
                               elitism = True,
                               maximise_fitness = True)

def my_create_individual(data):
    return [random.randint(0, 15) for _ in range(len(data))]
ga.create_individual = my_create_individual

def aptidao(individual, data):
    dinheiro = 0
    peso = 0
    for quantidade, caixa in zip(individual, data):
        dinheiro += quantidade * caixa['value']
        peso += quantidade * caixa['weight']
    if peso > 15:
        return 0
    return dinheiro
ga.fitness_function = aptidao

def crossover(parent_1, parent_2):
    corte = random.randrange(1, len(parent_1))
    child_1 = parent_1[:corte] + parent_2[corte:]
    child_2 = parent_2[:corte] + parent_1[corte:]
    return child_1, child_2
ga.crossover_function = crossover

def my_mutation(individual):
    posicao = random.randrange(len(individual))
    individual[posicao] = random.randint(0, 15)
ga.mutate_function = my_mutation

def my_selection(population):
    competidores = random.sample(population, 3)
    return max(competidores, key=lambda ind: ind.fitness)
ga.selection_function = my_selection

ga.run()

print("Melhor solução GA (Mochila):", ga.best_individual())

Melhor solução GA (Mochila): (0, [6, 15, 13, 8, 4])


In [3]:
import pyswarms as ps
import numpy as np

data = [{'name': 'verde', 'value': 4, 'weight': 12},
        {'name': 'cinza', 'value': 2, 'weight': 1},
        {'name': 'amarelo', 'value': 10, 'weight': 4},
        {'name': 'laranja', 'value': 1, 'weight': 1},
        {'name': 'azul', 'value': 2, 'weight': 2}]

def aptidao_pso(enxame, data=data):
    resultados = []
    for particula in enxame:
        individual = np.round(particula).astype(int)
        individual = np.maximum(0, individual)

        dinheiro, peso = 0, 0
        for quantidade, caixa in zip(individual, data):
            dinheiro += quantidade * caixa['value']
            peso += quantidade * caixa['weight']

        if peso > 15:
            dinheiro = 0

        resultados.append(-dinheiro)

    return np.array(resultados)

bounds = (np.zeros(5), np.full(5, 15))
options = {'c1': 1.5, 'c2': 1.5, 'w': 0.7}

pso = ps.single.GlobalBestPSO(n_particles=30, dimensions=5, options=options, bounds=bounds)

melhor_custo, melhor_individuo = pso.optimize(aptidao_pso, iters=50)

individuo_final = np.round(melhor_individuo).astype(int)
print("Melhor solução PSO (Quantidades):", individuo_final)
print("Dinheiro obtido:", -melhor_custo)


2026-08-29 21:57:43,985 - pyswarms.single.global_best - INFO - Optimize for 50 iters with {'c1': 1.5, 'c2': 1.5, 'w': 0.7}
pyswarms.single.global_best: 100%|██████████|50/50, best_cost=0
2026-08-29 21:57:44,105 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.0, best pos: [ 6.34708796 10.77466107 13.502563   12.65785817  7.24968544]


Melhor solução PSO (Quantidades): [ 6 11 14 13  7]
Dinheiro obtido: -0.0


In [5]:
from pyeasyga import pyeasyga
import random

# O pyeasyga exige um parâmetro 'data' na inicialização.
# Vamos usar isso apenas para passar os limites do nosso problema (-10 a 10).
limites = (-10, 10)

# Aumentamos a população e gerações porque buscar números decimais exige mais tentativas
ga = pyeasyga.GeneticAlgorithm(seed_data=limites,
                               population_size=100,
                               generations=200,
                               crossover_probability=0.9,
                               mutation_probability=0.3,
                               elitism=True,
                               maximise_fitness=True)

# 1. CRIAR INDIVÍDUO: Uma lista com 3 números reais (x, y, z) sorteados no intervalo.
def my_create_individual(data):
    min_val, max_val = data
    return [random.uniform(min_val, max_val) for _ in range(3)]
ga.create_individual = my_create_individual

# 2. FUNÇÃO DE APTIDÃO (FITNESS): A equação x² + y² + z²
# Como queremos que o resultado seja 0 e o GA tenta sempre puxar a nota para cima (maximizar),
# nós retornamos o valor negativo da conta. Assim, o teto máximo que a nota consegue atingir é o próprio 0.
def aptidao(individual, data):
    x, y, z = individual
    resultado = x**2 + y**2 + z**2
    return -resultado
ga.fitness_function = aptidao

# 3. MUTAÇÃO: Escolhe uma das três variáveis (x, y ou z) e joga um número novo lá dentro.
def my_mutation(individual):
    posicao = random.randrange(len(individual))
    individual[posicao] = random.uniform(-10, 10)
ga.mutate_function = my_mutation

ga.run()

melhor_nota, melhor_individuo = ga.best_individual()

# Retiramos o sinal negativo para exibir o resultado real da equação
print(f"Resultado final da equação f(x,y,z): {-melhor_nota:.6f}")
print(f"Valores encontrados: x = {melhor_individuo[0]:.4f} | y = {melhor_individuo[1]:.4f} | z = {melhor_individuo[2]:.4f}")

Resultado final da equação f(x,y,z): 0.000085
Valores encontrados: x = 0.0059 | y = -0.0063 | z = 0.0034


In [6]:
import pyswarms as ps
import numpy as np

# 1. FUNÇÃO DE APTIDÃO: PySwarms já tenta achar o menor valor possível.
# O enxame é uma matriz onde cada linha é uma partícula [x, y, z].
def aptidao_raizes_pso(enxame):
    # Elevamos todos os valores ao quadrado e somamos (x² + y² + z²) para cada partícula
    resultados = np.sum(enxame**2, axis=1)
    return resultados

# 2. LIMITES: x, y, z precisam estar entre -10 e 10
limite_inferior = np.array([-10.0, -10.0, -10.0])
limite_superior = np.array([10.0, 10.0, 10.0])
bounds = (limite_inferior, limite_superior)

options = {'c1': 1.5, 'c2': 1.5, 'w': 0.7}

# Configurando o enxame com 3 dimensões (x, y, z)
pso_raizes = ps.single.GlobalBestPSO(n_particles=50, dimensions=3, options=options, bounds=bounds)

# Rodando a otimização
melhor_custo, melhor_posicao = pso_raizes.optimize(aptidao_raizes_pso, iters=100)

print(f"Resultado final da equação f(x,y,z): {melhor_custo:.10f}")
print(f"Valores encontrados: x = {melhor_posicao[0]:.6f} | y = {melhor_posicao[1]:.6f} | z = {melhor_posicao[2]:.6f}")

2026-08-29 22:24:08,129 - pyswarms.single.global_best - INFO - Optimize for 100 iters with {'c1': 1.5, 'c2': 1.5, 'w': 0.7}
pyswarms.single.global_best: 100%|██████████|100/100, best_cost=8.68e-13
2026-08-29 22:24:08,279 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 8.681514179784229e-13, best pos: [ 9.08641277e-07  1.86452793e-07 -8.80783961e-08]


Resultado final da equação f(x,y,z): 0.0000000000
Valores encontrados: x = 0.000001 | y = 0.000000 | z = -0.000000
